# Crawler Member A trên Kaggle

Notebook này lấy cách chạy đồng thời và giới hạn thời gian từ runner Member B trên nhánh Huyen, nhưng cấu hình owner a và toàn bộ 44 hostname Member A. Trước khi chạy Kaggle, hãy đẩy các thay đổi crawler lên nhánh Huyen; notebook cài crawler từ nhánh này để dùng cập nhật xử lý redirect robots.txt. Notebook tự xác minh links_corpus.parquet theo SHA-256 của snapshot BTC mới nhất đã ghim trên Hugging Face. Nếu file Kaggle đính kèm là snapshot cũ, notebook tự tải đúng bản mới (1.194.469 URL); query.parquet không được dùng để crawl.

Không đặt OFFSET/BATCH_SIZE: SQLite tự quản lý toàn bộ URL A còn chờ. Mỗi phiên tự dừng mềm sau 10 giờ; kết quả và trạng thái được lưu vào checkpoint, rồi đóng gói thành member_a_resume.zip. Lần sau thêm ZIP (hoặc thư mục đã bung ZIP) làm Input; notebook tự nhận cả hai dạng. Khi chưa có checkpoint, notebook nạp document đã xuất bản từ Hugging Face và chỉ khớp theo ID + URL.

Tất cả 44 domain A đều được lập chỉ mục, kể cả các domain từng nằm trong danh sách loại trừ. Mặc định tối đa 4 domain chạy đồng thời và tối thiểu 5 giây giữa request cùng domain (hoặc crawl-delay dài hơn trong robots.txt; riêng VOV công bố 20 giây). Với baohaiphong.vn, hanoimoi.vn, vov.vn và youmed.vn, notebook kiểm tra sitemap HTTPS đã xác minh, giới hạn số tệp/kích thước, chỉ giữ URL cùng host và chỉ ưu tiên URL đã có trong links_corpus.parquet. Sitemap không cấp thêm quyền crawl và URL ngoài corpus không được thêm mới. API chỉ dùng khi nhà xuất bản công khai/tài liệu hóa và cho phép. Nếu host trả HTTP 403/429, captcha hoặc robots.txt không xác minh được, host đó được dừng an toàn và URL còn lại vẫn ở checkpoint.

Notebook không thể bảo đảm website sẽ không chặn crawler. Hãy xem member_a_domain_report.csv để khảo sát từng domain, bao gồm trạng thái sitemap và số URL khớp với corpus; với host từ chối truy cập, cần xin phép hoặc dùng API/dump được nhà xuất bản cung cấp thay vì đổi IP, proxy hay giả mạo trình duyệt.

Lưu ý: checkpoint rỗng tạo từ lần chạy snapshot cũ sẽ tự được làm sạch metadata trước khi lập chỉ mục snapshot chuẩn. Notebook dừng nếu sau bước lập chỉ mục vẫn không có URL A; sẽ không khởi động crawl với tổng 0 URL.


In [ ]:
%pip -q install "git+https://github.com/deckardLong/multilingual-medical-retrieval.git@Huyen#subdirectory=tasks/01_corpus_collection/crawler"


In [ ]:
from __future__ import annotations

import asyncio
import csv
import gzip
import hashlib
import io
import json
import os
import shutil
import sqlite3
import time
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath
from urllib.parse import quote, urljoin, urlsplit
from urllib.request import Request, urlopen

import aiohttp
import pyarrow.parquet as pq
import zstandard

from corpus_crawler.extract import make_records
from corpus_crawler.fetch import RespectfulFetcher
from corpus_crawler.output import ShardWriter
from corpus_crawler.settings import Settings
from corpus_crawler.state import CrawlState
from corpus_crawler.text import normalize_hostname

OWNER = "a"
HUGGINGFACE_REPO = "ai-guru-medical-retrieval/multilingual-medical-retrieval-corpus"
DATASET_REVISION = "0a797cc147fb7706e4f99495fd87fada3747ea6e"
LINKS_SHA256 = "9e34cb4b7d6424b80ed2634ada581ea9e2d97da69c245d80bdbb31413a78e8ce"
INPUT_MTIME_NS = 1_700_000_000_000_000_000
CRAWLER_REVISION = "Huyen"
MAX_RUNTIME_HOURS = 10.0
GLOBAL_CONCURRENCY = 4
REQUESTS_PER_DOMAIN = 0.2
SHARD_DOCS = 20_000
PROGRESS_SECONDS = 5
USER_AGENT = "R2AI2026CorpusResearch/1.0 (respectful multi-domain crawler; Member A)"
SITEMAP_SEEDS = {
    "baohaiphong.vn": ("https://baohaiphong.vn/sitemap.xml",),
    "hanoimoi.vn": ("https://hanoimoi.vn/sitemap.xml",),
    "vov.vn": ("https://vov.vn/sitemap.xml", "https://vov.vn/news-sitemap.xml"),
    "youmed.vn": ("https://youmed.vn/sitemap_index.xml", "https://youmed.vn/tin-tuc/sitemap_index.xml"),
}
MAX_SITEMAP_FILES_PER_DOMAIN = 30
MAX_SITEMAP_URLS_PER_DOMAIN = 100_000
MAX_SITEMAP_BYTES = 10_000_000

DOMAINS_A = frozenset("""
cnkang.com ask.39.net zysjonline.com zhongyibaodian.net nhathuoclongchau.com.vn
zydcd.com youlai.cn laodong.vn phunusuckhoe.giadinhonline.vn bingli.iiyi.com
qihuangzhishu.com baohaiphong.vn vietnamnet.vn test.pmphai.com tiemchunglongchau.com.vn
baocantho.com.vn woman.39.net baoangiang.com.vn vov.vn khoahocphothong.vn youmed.vn
nk.39.net msdmanuals.cn article.iiyi.com baoquangtri.vn man.39.net baovinhlong.com.vn
tamanhhospital.vn tnb.39.net tuoitre.vn qdnd.vn baogialai.com.vn ek.39.net
baoquangninh.vn baidu.com baochinhphu.vn vnexpress.net sggp.org.vn pf.39.net
health.people.com.cn hanoimoi.vn baotayninh.vn byby.39.net benhvienvietduc.org
""".split())

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
CORPUS_ROOT = WORK_ROOT / "corpus"
VIZ_ROOT = CORPUS_ROOT / "viz"
HF_LINKS_PATH = CORPUS_ROOT / "raw" / "links_corpus.parquet"
STATE_PATH = VIZ_ROOT / "state-a-small.sqlite3"
LEGACY_STATE_PATH = WORK_ROOT / "state-a-current.sqlite3"
RESUME_BUNDLE = WORK_ROOT / "member_a_resume.zip"
DOMAIN_REPORT = WORK_ROOT / "member_a_domain_report.csv"
for directory in (VIZ_ROOT / "docs" / OWNER, VIZ_ROOT / "crawl_log" / OWNER, VIZ_ROOT / "manifest" / OWNER):
    directory.mkdir(parents=True, exist_ok=True)

# Khôi phục bundle dù Kaggle giữ ZIP nguyên trạng hay tự bung thành thư mục.
def copy_resume_outputs(source_viz: Path) -> None:
    for name in ("docs", "crawl_log", "manifest"):
        source = source_viz / name / OWNER
        if source.is_dir():
            target = VIZ_ROOT / name / OWNER
            shutil.copytree(source, target, dirs_exist_ok=True)

if not STATE_PATH.exists():
    bundles = sorted(INPUT_ROOT.rglob("member_a_resume.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    if bundles:
        with zipfile.ZipFile(bundles[0]) as archive:
            for member in archive.infolist():
                item = PurePosixPath(member.filename)
                if item.is_absolute() or ".." in item.parts:
                    continue
                if item.as_posix() == "state-a-current.sqlite3":
                    target = LEGACY_STATE_PATH
                elif item.parts[:2] == ("corpus", "viz"):
                    target = WORK_ROOT.joinpath(*item.parts)
                elif item.parts[:2] in {("docs", "a"), ("crawl_log", "a"), ("manifest", "a")}:
                    target = VIZ_ROOT.joinpath(*item.parts)
                else:
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
        print(f"Đã khôi phục checkpoint từ ZIP: {bundles[0]}")
    else:
        extracted_states = sorted(
            (path for name in ("state-a-small.sqlite3", "state-a-current.sqlite3", "state-a.sqlite3") for path in INPUT_ROOT.rglob(name)),
            key=lambda p: (p.name != "state-a-small.sqlite3", str(p)),
        )
        if extracted_states:
            source_state = extracted_states[0]
            if source_state.name == "state-a-small.sqlite3":
                shutil.copy2(source_state, STATE_PATH)
                source_viz = source_state.parent
            elif source_state.parent.name == "viz":
                shutil.copy2(source_state, STATE_PATH)
                source_viz = source_state.parent
            else:
                shutil.copy2(source_state, LEGACY_STATE_PATH)
                source_viz = source_state.parent / "corpus" / "viz"
            copy_resume_outputs(source_viz)
            print(f"Đã khôi phục checkpoint đã giải nén: {source_state}")

link_candidates = sorted(INPUT_ROOT.rglob("links_corpus.parquet"))
HF_LINKS_PATH.parent.mkdir(parents=True, exist_ok=True)
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

matching_input = next((p for p in link_candidates if sha256_file(p) == LINKS_SHA256), None)
if matching_input is not None:
    if matching_input.resolve() != HF_LINKS_PATH.resolve():
        shutil.copyfile(matching_input, HF_LINKS_PATH)
    print(f"Đã xác minh links_corpus.parquet đính kèm đúng snapshot BTC mới: {matching_input}")
else:
    if link_candidates:
        print("File links_corpus.parquet đính kèm không phải snapshot mới; tự tải bản chuẩn từ Hugging Face.")
    else:
        print("Không có links_corpus.parquet đính kèm; tự tải snapshot chuẩn từ Hugging Face.")
    links_url = f"https://huggingface.co/datasets/{HUGGINGFACE_REPO}/resolve/{DATASET_REVISION}/corpus/raw/links_corpus.parquet?download=true"
    download_path = HF_LINKS_PATH.with_suffix(".parquet.download")
    request = Request(links_url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=180) as response, download_path.open("wb") as output:
        shutil.copyfileobj(response, output)
    actual_sha256 = sha256_file(download_path)
    if actual_sha256 != LINKS_SHA256:
        download_path.unlink(missing_ok=True)
        raise RuntimeError(f"Snapshot links tải về không khớp SHA-256 chuẩn: {actual_sha256}")
    download_path.replace(HF_LINKS_PATH)
os.utime(HF_LINKS_PATH, ns=(INPUT_MTIME_NS, INPUT_MTIME_NS))
INPUT_PATH = HF_LINKS_PATH
links_file = pq.ParquetFile(INPUT_PATH)
if not {"id", "url"}.issubset(links_file.schema_arrow.names):
    raise ValueError("links_corpus.parquet phải có cột id và url.")
if links_file.metadata.num_rows != 1_194_469:
    raise ValueError(f"Snapshot links không đúng phiên bản đã xác minh: mong đợi 1.194.469 URL, nhận {links_file.metadata.num_rows:,}.")
query_candidates = sorted(INPUT_ROOT.rglob("query.parquet"))
if query_candidates:
    query_file = pq.ParquetFile(query_candidates[0])
    if not {"id", "query"}.issubset(query_file.schema_arrow.names):
        raise ValueError("query.parquet phải có cột id và query.")
    print(f"File query: {query_candidates[0]} | {query_file.metadata.num_rows:,} dòng")
print(f"Snapshot đầu vào: {INPUT_PATH} | {links_file.metadata.num_rows:,} URL")

settings = Settings(
    owner=OWNER,
    schema_version="1.0",
    norm_version="n1",
    input_parquet=INPUT_PATH,
    corpus_root=CORPUS_ROOT,
    global_concurrency=GLOBAL_CONCURRENCY,
    per_domain_concurrency=1,
    requests_per_domain=REQUESTS_PER_DOMAIN,
    timeout_seconds=25,
    max_response_bytes=5_000_000,
    max_retries=3,
    retry_backoff_seconds=2.0,
    min_text_chars=200,
    shard_docs=SHARD_DOCS,
    progress_seconds=PROGRESS_SECONDS,
    user_agent=USER_AGENT,
    code_commit=f"Huyen-{CRAWLER_REVISION[:7]}",
    domains=DOMAINS_A,
)
state = CrawlState(STATE_PATH)
if state.totals()[0] == 0:
    with state.connection:
        state.connection.execute("DELETE FROM metadata WHERE key IN ('input_fingerprint','source_seeded','hf_docs_imported')")
    print("Checkpoint chưa có URL nào; đã xóa metadata rỗng cũ để nạp lại snapshot chuẩn.")
try:
    added, scanned = state.prepare_source(settings, None)
except ValueError as exc:
    state.close()
    if "No source URLs matched" in str(exc):
        raise RuntimeError("Snapshot links không có URL Member A. Đã dừng trước khi gửi request; hãy đính kèm links_corpus.parquet đúng snapshot mới.") from exc
    raise
total_urls, done_urls, pending_urls = state.totals()
if total_urls == 0:
    state.close()
    raise RuntimeError("Đã dừng an toàn: snapshot chuẩn vẫn không có URL Member A; không gửi request crawl nào.")
registered_domains = {row[0] for row in state.connection.execute("SELECT DISTINCT domain FROM urls")}
print(f"Đã lập chỉ mục {total_urls:,} URL của {len(registered_domains):,}/44 domain A | mới {added:,} | đã xong {done_urls:,} | còn {pending_urls:,}")

# Nhập checkpoint notebook cũ nếu người dùng đính kèm bundle/state cũ.
legacy_candidates = [LEGACY_STATE_PATH]
legacy_candidates.extend(INPUT_ROOT.rglob("state-a-current.sqlite3"))
legacy_candidates.extend(INPUT_ROOT.rglob("state-a.sqlite3"))
legacy_path = next((p for p in legacy_candidates if p.exists() and p.resolve() != STATE_PATH.resolve()), None)
if legacy_path and state.connection.execute("SELECT 1 FROM metadata WHERE key='legacy_state_imported'").fetchone() is None:
    legacy = sqlite3.connect(f"file:{legacy_path}?mode=ro", uri=True)
    try:
        tables = {r[0] for r in legacy.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if "urls" in tables:
            columns = {r[1] for r in legacy.execute("PRAGMA table_info(urls)")}
            needed = {"id", "url", "status", "attempt", "log_json"}
            if needed.issubset(columns):
                doc_expr = "doc_json" if "doc_json" in columns else "NULL"
                log_exported_expr = "log_exported" if "log_exported" in columns else "0"
                doc_exported_expr = "doc_exported" if "doc_exported" in columns else "0"
                shard_expr = "doc_shard" if "doc_shard" in columns else "NULL"
                old_rows = legacy.execute(f"SELECT id,url,attempt,log_json,{doc_expr},{log_exported_expr},{doc_exported_expr},{shard_expr} FROM urls WHERE status='done'")
                migrated = 0
                while True:
                    rows = old_rows.fetchmany(2_000)
                    if not rows:
                        break
                    with state.connection:
                        cursor = state.connection.executemany("""
                            UPDATE urls SET status='done',attempt=?,log_json=?,doc_json=?,log_exported=?,doc_exported=?,doc_shard=?
                            WHERE id=? AND url=? AND status='pending'
                        """, [(r[2],r[3],r[4],int(r[5] or 0),int(r[6] or 0),r[7],r[0],r[1]) for r in rows])
                        migrated += cursor.rowcount
                for hash_table in ("content_hashes", "hashes"):
                    if hash_table in tables:
                        hash_columns = {r[1] for r in legacy.execute(f"PRAGMA table_info({hash_table})")}
                        if {"content_hash", "uid"}.issubset(hash_columns):
                            for digest, uid in legacy.execute(f"SELECT content_hash,uid FROM {hash_table}"):
                                digest = str(digest)
                                if not digest.startswith("sha1:"):
                                    digest = "sha1:" + digest
                                state.connection.execute("INSERT OR IGNORE INTO content_hashes(content_hash,uid) VALUES(?,?)", (digest,uid))
                        break
                state.connection.execute("INSERT OR REPLACE INTO metadata(key,value) VALUES('legacy_state_imported',?)", (str(legacy_path),))
                state.connection.commit()
                print(f"Đã nhập {migrated:,} kết quả checkpoint cũ (đối chiếu chính xác ID + URL).")
    finally:
        legacy.close()

# Hugging Face hiện chỉ có docs/a, không có crawl_log/a; chỉ resume document có ID + URL khớp.
if state.connection.execute("SELECT 1 FROM metadata WHERE key='hf_docs_imported'").fetchone() is None:
    hf_api = f"https://huggingface.co/api/datasets/{HUGGINGFACE_REPO}/tree/{DATASET_REVISION}/corpus/viz/docs/a?recursive=true&expand=false"
    hf_base = f"https://huggingface.co/datasets/{HUGGINGFACE_REPO}/resolve/{DATASET_REVISION}/"
    request = Request(hf_api, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=60) as response:
        hf_files = json.load(response)
    shard_paths = sorted(item["path"] for item in hf_files if item.get("type") == "file" and item.get("path", "").startswith("corpus/viz/docs/a/") and item["path"].endswith(".jsonl.zst"))
    if not shard_paths:
        state.close()
        raise RuntimeError("Không tìm thấy shard docs/a trên Hugging Face; không thể xác minh lịch sử resume.")
    hf_seen = 0
    hf_matched = 0
    pending_batch = []
    def flush_hf_batch() -> None:
        global hf_matched
        if not pending_batch:
            return
        with state.connection:
            cursor = state.connection.executemany("""
                UPDATE urls SET status='done',attempt=0,log_json=?,doc_json=?,log_exported=0,doc_exported=1,doc_shard=?
                WHERE id=? AND url=? AND status='pending'
            """, [(item[0],item[1],item[2],item[3],item[4]) for item in pending_batch])
            hf_matched += cursor.rowcount
            state.connection.executemany("INSERT OR IGNORE INTO content_hashes(content_hash,uid) VALUES(?,?)", [(item[5],item[6]) for item in pending_batch if item[5] and item[6]])
        pending_batch.clear()
    for shard_path in shard_paths:
        shard_url = hf_base + quote(shard_path, safe="/") + "?download=true"
        shard_tmp = WORK_ROOT / ("hf-" + Path(shard_path).name + ".download")
        request = Request(shard_url, headers={"User-Agent": USER_AGENT})
        with urlopen(request, timeout=120) as response, shard_tmp.open("wb") as output:
            shutil.copyfileobj(response, output)
        try:
            with shard_tmp.open("rb") as compressed, zstandard.ZstdDecompressor().stream_reader(compressed) as reader:
                text_reader = io.TextIOWrapper(reader, encoding="utf-8")
                for line in text_reader:
                    doc = json.loads(line)
                    if doc.get("owner") != OWNER or doc.get("source") != "btc_url":
                        continue
                    try:
                        source_id = int(doc["doc_id"])
                    except (KeyError, TypeError, ValueError):
                        continue
                    source_url = doc.get("url")
                    fetch = doc.get("fetch") or {}
                    status = fetch.get("status")
                    if not isinstance(source_url, str) or status not in {"ok", "duplicate", "lang_mismatch"}:
                        continue
                    current = state.connection.execute("SELECT url,domain FROM urls WHERE id=?", (source_id,)).fetchone()
                    if current is None or current["url"] != source_url:
                        continue
                    digest = str(doc.get("content_hash") or "")
                    uid = str(doc.get("uid") or "")
                    digest = digest[5:] if digest.startswith("sha1:") else digest
                    log = {"uid":uid or f"src:{source_id}","id":source_id,"url":source_url,"domain":current["domain"],"owner":OWNER,"status":status,"http_status":fetch.get("http_status"),"attempt":0,"fetched_at":fetch.get("fetched_at"),"bytes":0,"final_url":source_url,"error":None,"shard":"huggingface"}
                    pending_batch.append((json.dumps(log,ensure_ascii=False),json.dumps(doc,ensure_ascii=False),Path(shard_path).name,source_id,source_url,"sha1:"+digest if digest else None,uid))
                    hf_seen += 1
                    if len(pending_batch) >= 1_000:
                        flush_hf_batch()
            print(f"Đã đọc lịch sử Hugging Face: {Path(shard_path).name} | đã khớp {hf_matched:,} document")
        finally:
            shard_tmp.unlink(missing_ok=True)
    flush_hf_batch()
    state.connection.execute("INSERT INTO metadata(key,value) VALUES('hf_docs_imported',?)", (json.dumps({"shards":len(shard_paths),"documents":hf_seen,"matched":hf_matched}),))
    state.connection.commit()
    print(f"Hugging Face: đối chiếu thành công {hf_matched:,}/{hf_seen:,} document theo ID + URL.")

writer = ShardWriter(settings, state)
with state.connection:
    state.connection.execute("CREATE TABLE IF NOT EXISTS sitemap_urls (domain TEXT NOT NULL, url TEXT NOT NULL, PRIMARY KEY(domain,url)) WITHOUT ROWID")
    state.connection.execute("""
        CREATE TABLE IF NOT EXISTS sitemap_runs (
            domain TEXT PRIMARY KEY, checked_at TEXT NOT NULL, status TEXT NOT NULL,
            sitemap_files INTEGER NOT NULL, discovered_urls INTEGER NOT NULL,
            matched_pending INTEGER NOT NULL, detail TEXT
        )
    """)
    state.connection.execute("""
        CREATE TABLE IF NOT EXISTS host_pauses (
            domain TEXT PRIMARY KEY, paused_at TEXT NOT NULL, reason TEXT NOT NULL,
            retry_after REAL
        )
    """)
    previous_blocked = state.connection.execute("""
        SELECT domain,json_extract(log_json,'$.error') AS error
        FROM urls
        WHERE status='done' AND json_extract(log_json,'$.status')='blocked'
    """).fetchall()
    for previous in previous_blocked:
        reason = str(previous["error"] or "")
        lower_reason = reason.lower()
        if not any(marker in lower_reason for marker in ("http 403", "http 429", "possible captcha", "robots.txt", "disallowed by robots")):
            continue
        if "robots.txt returned http 301" in lower_reason:
            retry_after = 0.0
        elif any(marker in lower_reason for marker in ("http 403", "http 429", "possible captcha", "disallowed by robots")):
            retry_after = None
        else:
            retry_after = time.time() + 24 * 3600
        state.connection.execute("INSERT OR IGNORE INTO host_pauses(domain,paused_at,reason,retry_after) VALUES(?,datetime('now'),?,?)", (previous["domain"],reason,retry_after))
    state.connection.commit()

# Chỉ điền domain sau khi đã xử lý nguyên nhân và muốn thử lại thủ công.
FORCE_RECHECK_HOSTS: set[str] = set()
print(f"Bắt đầu Member A: tối đa {GLOBAL_CONCURRENCY} domain chạy đồng thời; không dùng OFFSET và không loại domain.")

blocked_host_markers = ("http 403", "http 429", "possible captcha", "robots.txt unavailable", "robots.txt returned http", "robots.txt redirect")
blocked_hosts: set[str] = set()
network_slots = asyncio.Semaphore(GLOBAL_CONCURRENCY)
session_counts: Counter[str] = Counter()
run_started = time.monotonic()
deadline = run_started + MAX_RUNTIME_HOURS * 3600
last_report = run_started

async def crawl_all_domains() -> None:
    report_clock = {"last": run_started}
    timeout = aiohttp.ClientTimeout(total=settings.timeout_seconds)
    connector = aiohttp.TCPConnector(limit=GLOBAL_CONCURRENCY)
    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers={"User-Agent":USER_AGENT}) as session:
        fetcher = RespectfulFetcher(settings, session)
        if FORCE_RECHECK_HOSTS:
            with state.connection:
                state.connection.executemany("DELETE FROM host_pauses WHERE domain=?", ((domain,) for domain in FORCE_RECHECK_HOSTS))
        pause_rows = state.connection.execute("SELECT domain,reason,retry_after FROM host_pauses").fetchall()
        now_epoch = time.time()
        active_pauses = {
            row["domain"]: (row["reason"], row["retry_after"])
            for row in pause_rows
            if row["retry_after"] is None or row["retry_after"] > now_epoch
        }
        expired_pauses = [row["domain"] for row in pause_rows if row["retry_after"] is not None and row["retry_after"] <= now_epoch]
        if expired_pauses:
            with state.connection:
                state.connection.executemany("DELETE FROM host_pauses WHERE domain=?", ((domain,) for domain in expired_pauses))
        blocked_hosts.update(active_pauses)
        for domain, (reason, retry_after) in sorted(active_pauses.items()):
            retry_text = "thử lại thủ công sau khi được phép" if retry_after is None else time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime(retry_after))
            print(f"\nBỏ qua {domain}: đang tạm dừng từ checkpoint ({reason}); {retry_text}.")
        async def discover_sitemap_urls(domain: str) -> None:
            seeds = SITEMAP_SEEDS.get(domain, ())
            if not seeds:
                return
            discovered: set[str] = set()
            sitemap_queue = list(seeds)
            seen_sitemaps: set[str] = set()
            sitemap_files = 0
            robots_delay = 0.0
            error_detail = None
            last_request = 0.0
            status = "ok"
            sitemap_errors = []
            try:
                while sitemap_queue and sitemap_files < MAX_SITEMAP_FILES_PER_DOMAIN and len(discovered) < MAX_SITEMAP_URLS_PER_DOMAIN:
                    sitemap_url = sitemap_queue.pop(0)
                    if sitemap_url in seen_sitemaps:
                        continue
                    seen_sitemaps.add(sitemap_url)
                    if normalize_hostname(sitemap_url) != domain or urlsplit(sitemap_url).scheme != "https":
                        continue
                    parser, robots_delay, robots_error = await fetcher._robots_policy(sitemap_url, domain)
                    if robots_error or parser is None:
                        status = "robots_unavailable"
                        error_detail = robots_error or "Could not verify robots.txt"
                        break
                    if not parser.can_fetch(sitemap_url, user_agent=USER_AGENT):
                        status = "partial_error"
                        sitemap_errors.append(f"robots.txt disallows sitemap URL: {sitemap_url}")
                        continue
                    request_gap = max(1.0 / REQUESTS_PER_DOMAIN, robots_delay)
                    wait_for = request_gap - (time.monotonic() - last_request)
                    if last_request and wait_for > 0:
                        await asyncio.sleep(wait_for)
                    async with session.get(sitemap_url, allow_redirects=False) as response:
                        last_request = time.monotonic()
                        if response.status in {301, 302, 303, 307, 308}:
                            location = response.headers.get("Location")
                            target = urljoin(sitemap_url, location) if location else ""
                            if location and normalize_hostname(target) == domain and urlsplit(target).scheme == "https" and parser.can_fetch(target, user_agent=USER_AGENT):
                                sitemap_queue.insert(0, target)
                                continue
                            status = "partial_error"
                            sitemap_errors.append(f"Sitemap redirect not followed: HTTP {response.status} -> {location or 'missing Location'}")
                            continue
                        if response.status != 200:
                            status = "partial_error"
                            sitemap_errors.append(f"Sitemap returned HTTP {response.status}: {sitemap_url}")
                            continue
                        payload_chunks = []
                        payload_size = 0
                        async for chunk in response.content.iter_chunked(64 * 1024):
                            payload_size += len(chunk)
                            if payload_size > MAX_SITEMAP_BYTES:
                                break
                            payload_chunks.append(chunk)
                        if payload_size > MAX_SITEMAP_BYTES:
                            status = "partial_error"
                            sitemap_errors.append(f"Sitemap exceeds {MAX_SITEMAP_BYTES:,} bytes: {sitemap_url}")
                            continue
                    payload = b"".join(payload_chunks)
                    if payload.startswith(b"\x1f\x8b"):
                        with gzip.GzipFile(fileobj=io.BytesIO(payload)) as compressed:
                            payload = compressed.read(MAX_SITEMAP_BYTES + 1)
                        if len(payload) > MAX_SITEMAP_BYTES:
                            status = "partial_error"
                            sitemap_errors.append(f"Expanded sitemap exceeds {MAX_SITEMAP_BYTES:,} bytes: {sitemap_url}")
                            continue
                    try:
                        root = ET.fromstring(payload)
                    except ET.ParseError as exc:
                        status = "partial_error"
                        sitemap_errors.append(f"{sitemap_url}: {exc}")
                        continue
                    root_kind = root.tag.rsplit("}", 1)[-1].lower()
                    sitemap_files += 1
                    for element in root.iter():
                        if element.tag.rsplit("}", 1)[-1].lower() != "loc" or not element.text:
                            continue
                        child_url = element.text.strip()
                        if normalize_hostname(child_url) != domain or urlsplit(child_url).scheme != "https":
                            continue
                        if root_kind == "sitemapindex":
                            if child_url not in seen_sitemaps and len(seen_sitemaps) + len(sitemap_queue) < MAX_SITEMAP_FILES_PER_DOMAIN:
                                sitemap_queue.append(child_url)
                        elif root_kind == "urlset":
                            if parser.can_fetch(child_url, user_agent=USER_AGENT):
                                discovered.add(child_url)
                                if len(discovered) >= MAX_SITEMAP_URLS_PER_DOMAIN:
                                    break
                    if root_kind not in {"sitemapindex", "urlset"}:
                        status = "partial_error"
                        sitemap_errors.append(f"Unexpected sitemap XML root: {root_kind or 'empty'} at {sitemap_url}")
                        continue
                if sitemap_queue and status == "ok":
                    status = "limit_reached"
                    error_detail = "Sitemap traversal stopped at configured per-domain limit."
            except (aiohttp.ClientError, asyncio.TimeoutError, OSError) as exc:
                status = "partial_error" if sitemap_files else "error"
                sitemap_errors.append(f"{type(exc).__name__}: {exc}")
            if sitemap_errors:
                error_detail = "; ".join(sitemap_errors[:20])
            matched = 0
            if status in {"ok", "limit_reached", "partial_error"}:
                with state.connection:
                    state.connection.execute("DELETE FROM sitemap_urls WHERE domain=?", (domain,))
                    state.connection.executemany("INSERT OR IGNORE INTO sitemap_urls(domain,url) VALUES(?,?)", ((domain, item) for item in discovered))
                    matched = state.connection.execute("""
                        SELECT COUNT(*) FROM urls u JOIN sitemap_urls s ON s.domain=u.domain AND s.url=u.url
                        WHERE u.domain=? AND u.status='pending'
                    """, (domain,)).fetchone()[0]
            with state.connection:
                state.connection.execute("""
                    INSERT OR REPLACE INTO sitemap_runs(domain,checked_at,status,sitemap_files,discovered_urls,matched_pending,detail)
                    VALUES(?,datetime('now'),?,?,?,?,?)
                """, (domain,status,sitemap_files,len(discovered),matched,error_detail))
            print(f"\nSitemap {domain}: trạng thái={status} | tệp={sitemap_files} | URL hợp lệ={len(discovered):,} | URL còn chờ trong corpus khớp={matched:,}" + (f" | {error_detail}" if error_detail else ""))
        for sitemap_domain in SITEMAP_SEEDS:
            if sitemap_domain in registered_domains and sitemap_domain not in blocked_hosts:
                await discover_sitemap_urls(sitemap_domain)
        sitemap_pending = state.connection.execute("SELECT COUNT(*) FROM sitemap_urls").fetchone()[0]
        print(f"Đã lập chỉ mục sitemap ưu tiên cho URL hiện có; sitemap URL đã ghi nhận={sitemap_pending:,}. URL ngoài links_corpus không được thêm vào corpus.")
        def claim_for_domain(domain: str):
            row = state.connection.execute("""
                SELECT u.id,u.url,u.domain FROM urls u
                WHERE u.status='pending' AND u.domain=?
                ORDER BY CASE WHEN EXISTS(SELECT 1 FROM sitemap_urls s WHERE s.domain=u.domain AND s.url=u.url) THEN 0 ELSE 1 END,u.id
                LIMIT 1
            """, (domain,)).fetchone()
            if row is None:
                return None
            with state.connection:
                cursor = state.connection.execute("UPDATE urls SET status='fetching' WHERE id=? AND status='pending'",(row["id"],))
            return row if cursor.rowcount == 1 else None
        async def report_progress() -> None:
            report_clock["last"] = time.monotonic()
            total, done, pending = state.totals()
            elapsed = max(report_clock["last"] - run_started, 0.001)
            rate = sum(session_counts.values()) / elapsed
            eta = pending / rate / 3600 if rate else 0.0
            counts = state.completed_counts()
            codes = (("blocked","bị chặn"),("too_short","nội dung ngắn"),("ok","đạt"),("empty","rỗng"),("http_error","lỗi HTTP"),("timeout","quá giờ"),("lang_mismatch","sai ngôn ngữ"),("duplicate","trùng lặp"))
            detail = " | ".join(f"{label} {counts.get(code,0):,}" for code,label in codes)
            print(f"\rMember A: {done:,}/{total:,} | còn {pending:,} | {detail} | {rate:.2f} URL/s | ETA {eta:.1f} giờ",end="",flush=True)
        async def crawl_domain(domain: str) -> None:
            while domain not in blocked_hosts:
                if time.monotonic() >= deadline:
                    return
                row = claim_for_domain(domain)
                if row is None:
                    return
                try:
                    async with network_slots:
                        result = await fetcher.fetch(str(row["url"]),domain)
                    log_record, doc_record = await asyncio.to_thread(make_records,int(row["id"]),str(row["url"]),domain,result,settings)
                    state.save_result(int(row["id"]),result.attempts,log_record,doc_record)
                    session_counts[log_record["status"]] += 1
                    writer.export_ready()
                    now = time.monotonic()
                    if now - report_clock["last"] >= PROGRESS_SECONDS:
                        await report_progress()
                    error = (result.error or "").lower()
                    if result.status in {"blocked", "http_error"} and any(marker in error for marker in blocked_host_markers):
                        blocked_hosts.add(domain)
                        permanent_pause = any(marker in error for marker in ("http 403", "http 429", "possible captcha", "disallowed by robots"))
                        retry_after = None if permanent_pause else time.time() + 7 * 24 * 3600
                        with state.connection:
                            state.connection.execute("INSERT OR REPLACE INTO host_pauses(domain,paused_at,reason,retry_after) VALUES(?,datetime('now'),?,?)", (domain, result.error or "blocked", retry_after))
                        retry_text = "thử lại thủ công sau khi được phép" if retry_after is None else "thử lại sau 7 ngày nếu lỗi không lặp lại"
                        print(f"\nTạm dừng {domain}: {result.error or 'block/robots'}; URL còn lại được giữ để resume; {retry_text}.")
                except asyncio.CancelledError:
                    raise
        tasks = [asyncio.create_task(crawl_domain(domain)) for domain in sorted(registered_domains)]
        try:
            while tasks:
                if time.monotonic() >= deadline:
                    print("\nĐã đến giới hạn 10 giờ; dừng mềm, xuất checkpoint và giữ URL chưa xử lý.")
                    break
                done_tasks, pending_tasks = await asyncio.wait(tasks,timeout=1.0,return_when=asyncio.FIRST_EXCEPTION)
                for task in done_tasks:
                    task.result()
                tasks = list(pending_tasks)
        except BaseException:
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks,return_exceptions=True)
            raise
        finally:
            for task in tasks:
                if not task.done():
                    task.cancel()
            if tasks:
                await asyncio.gather(*tasks,return_exceptions=True)
            writer.export_ready(force=True)

try:
    await crawl_all_domains()
except KeyboardInterrupt:
    print("\nĐã nhận lệnh dừng; các URL đang xử lý sẽ được đưa về chờ khi resume.")
finally:
    try:
        writer.export_ready(force=True)
        total_urls, done_urls, pending_urls = state.totals()
        print(f"\nTổng kết A: {done_urls:,}/{total_urls:,} đã xong | còn {pending_urls:,} | trạng thái {state.completed_counts()}")
        domain_rows = state.connection.execute("""
            SELECT domain,
                   SUM(CASE WHEN status='pending' THEN 1 ELSE 0 END) AS pending_urls,
                   SUM(CASE WHEN status='done' AND json_extract(log_json,'$.status')='blocked' THEN 1 ELSE 0 END) AS blocked_urls,
                   SUM(CASE WHEN status='done' AND json_extract(log_json,'$.status')='http_error' THEN 1 ELSE 0 END) AS http_errors,
                   GROUP_CONCAT(DISTINCT CASE WHEN status='done' AND json_extract(log_json,'$.status') IN ('blocked','http_error') THEN json_extract(log_json,'$.error') END) AS access_errors,
                   (SELECT COUNT(*) FROM sitemap_urls s WHERE s.domain=urls.domain) AS sitemap_urls_found,
                   (SELECT COUNT(*) FROM urls u JOIN sitemap_urls s ON s.domain=u.domain AND s.url=u.url WHERE u.domain=urls.domain AND u.status='pending') AS sitemap_pending_matches,
                   (SELECT status FROM sitemap_runs r WHERE r.domain=urls.domain) AS sitemap_status,
                   (SELECT detail FROM sitemap_runs r WHERE r.domain=urls.domain) AS sitemap_detail,
                   (SELECT reason FROM host_pauses p WHERE p.domain=urls.domain) AS host_pause_reason,
                   (SELECT retry_after FROM host_pauses p WHERE p.domain=urls.domain) AS host_retry_after
            FROM urls GROUP BY domain ORDER BY domain
        """).fetchall()
        report_records = []
        for row in domain_rows:
            record = dict(row)
            error_text = (record["access_errors"] or "").lower()
            if "disallowed by robots" in error_text:
                recommendation = "Bỏ qua URL bị robots disallow; chỉ dùng route khác nếu robots.txt cho phép."
            elif any(marker in error_text for marker in ("robots.txt unavailable", "robots.txt returned http", "http 403", "http 429", "captcha", "access challenge")):
                recommendation = "Dừng crawl host; xin phép hoặc dùng API/dataset công khai được nhà xuất bản cho phép."
            elif record["sitemap_status"] not in {None, "ok", "limit_reached", "partial_error"}:
                recommendation = "Sitemap không đọc được; xem sitemap_detail và chỉ thử endpoint công khai/được robots cho phép."
            elif record["sitemap_status"] == "partial_error":
                recommendation = "Đã lưu URL từ sitemap đọc được; xem sitemap_detail để biết phần còn thiếu."
            elif record["http_errors"]:
                recommendation = "Kiểm tra access_errors; giảm tốc độ hoặc dừng nếu host tiếp tục từ chối."
            else:
                recommendation = "HTML trực tiếp; sitemap chỉ để tìm URL; API/trình duyệt chỉ khi được công khai và cho phép."
            record["method_recommendation"] = recommendation
            report_records.append(record)
        with DOMAIN_REPORT.open("w", encoding="utf-8-sig", newline="") as report_file:
            report_writer = csv.DictWriter(report_file, fieldnames=("domain", "pending_urls", "blocked_urls", "http_errors", "access_errors", "sitemap_urls_found", "sitemap_pending_matches", "sitemap_status", "sitemap_detail", "host_pause_reason", "host_retry_after", "method_recommendation"))
            report_writer.writeheader()
            report_writer.writerows(report_records)
        print(f"Host tạm dừng do block/robots: {sorted(blocked_hosts)}")
        print(f"Báo cáo khảo sát domain: {DOMAIN_REPORT}")
    finally:
        with state.connection:
            state.connection.execute("UPDATE urls SET status='pending' WHERE status='fetching'")
        state.connection.execute("PRAGMA wal_checkpoint(FULL)")
        state.close()
        temp_bundle = RESUME_BUNDLE.with_suffix(".zip.tmp")
        with zipfile.ZipFile(temp_bundle,"w",compression=zipfile.ZIP_DEFLATED) as archive:
            files = [STATE_PATH, DOMAIN_REPORT]
            for folder in (VIZ_ROOT / "docs" / OWNER,VIZ_ROOT / "crawl_log" / OWNER,VIZ_ROOT / "manifest" / OWNER):
                files.extend(p for p in folder.rglob("*") if p.is_file() and not p.name.endswith(".tmp"))
            for p in sorted(set(files)):
                archive.write(p,arcname=p.relative_to(WORK_ROOT).as_posix())
        temp_bundle.replace(RESUME_BUNDLE)
        print(f"Đã lưu bundle resume: {RESUME_BUNDLE}")
